In [ ]:
import os
from dotenv import load_dotenv
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

print("\n--- NGÀY 95: CONVERSATIONAL RAG (RAG CÓ TRÍ NHỚ) ---")

load_dotenv()
os.environ["GOOGLE_API_KEY"] = os.getenv("GEMINI_API_KEY")

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.7)
# Sửa lại dòng này
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-04")

print("[1] Kết nối Database...")
vector_db = Chroma(persist_directory="./database", embedding_function=embeddings)
nguoi_tim_kiem = vector_db.as_retriever(search_kwargs={"k":3})

# 1. PROMPT VÀ DÂY CHUYỀN CHÍNH (Đã nâng cấp để nhận Lịch sử Chat)
mau_lenh = """
Bạn là một trợ lý AI thông minh. Hãy trả lời câu hỏi dựa TRÊN TÀI LIỆU được cung cấp.
Nếu không có trong tài liệu, hãy nói không biết.

TÀI LIỆU:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ('system', mau_lenh),
    MessagesPlaceholder(variable_name='history'),
    ('human', '{input}')
])

def format_docs(docs):
    return "\n\n".join(str(doc.page_content) for doc in docs if hasattr(doc, 'page_content'))

day_chuyen_rag = (
    {
        'context': nguoi_tim_kiem | format_docs, 
        'input': lambda x:x['input'], 
        'history': lambda x:x['history']
    }
    |prompt
    |llm
    |StrOutputParser()
)

#Bọc dây chuyền bằng lớp quản lý trí nhớ
#Nơi lưu trữ bộ nhớ tạm (Dictionary)
store = {}

def get_session_id(session_id: str):
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]

# Bao bọc day_chuyen_rag bằng RunnableWithMessageHistory
# Thằng này sẽ tự động lấy câu hỏi, nhét lịch sử vào 'chat_history', gửi cho chain xử lý, rồi lưu câu trả lời lại.
day_chuyen_rag_with_memory = RunnableWithMessageHistory(
    day_chuyen_rag,
    get_session_id,
    input_message_key = 'input',
    history_message_key = 'history',
)

#Thử nghiệm vòng lặp chat
print("\n[HỆ THỐNG]: Đã sẵn sàng trò chuyện về cuốn sách của bạn! (Gõ 'thoát' để dừng)\n")
session_id = "phien_chat_cua_minh" # ID phiên chat (quan trọng khi làm web cho nhiều người dùng)

while True:
    cau_hoi = input('Bạn: ')
    if cau_hoi.lower() in ['thoát', 'exit']:
        break
    
        # Thêm dòng này để debug
    docs = nguoi_tim_kiem.invoke(cau_hoi)
    print(f"DEBUG - Số lượng tài liệu tìm thấy: {len(docs)}")
    for i, d in enumerate(docs):
        print(f"Doc {i} type: {type(d.page_content)}")

    ket_qua = day_chuyen_rag_with_memory.invoke(
        {"input": cau_hoi},
        config={"configurable": {"session_id": session_id}}
    )

    print(f"[JARVIS]: {ket_qua}\n")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



--- NGÀY 95: CONVERSATIONAL RAG (RAG CÓ TRÍ NHỚ) ---
[1] Kết nối Database...

[HỆ THỐNG]: Đã sẵn sàng trò chuyện về cuốn sách của bạn! (Gõ 'thoát' để dừng)

DEBUG - Số lượng tài liệu tìm thấy: 0


GoogleGenerativeAIError: Error embedding content: file uri and mime_type are required.